In [1]:
#!source /nflvenv/bin/activate

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder

##Reading

In [3]:
df = pd.read_csv("train.csv")

/var/folders/qb/ldt_bc3j77969pkfrmpl2bbc0000gn/T/ipykernel_22402/2292805398.py:1: DtypeWarning: Columns (47) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("train.csv")


In [4]:
print("Shape:", df.shape)
print("Unique plays:", df["PlayId"].nunique())
print("Unique games:", df["GameId"].nunique())
print("Temps:", sorted(df["Season"].unique()))
df.head()

Shape: (682154, 49)
Unique plays: 31007
Unique games: 688
Temps: [np.int64(2017), np.int64(2018), np.int64(2019)]


,GameId,PlayId,Team,X,Y,S,A,Dis,Orientation,Dir,...,Week,Stadium,Location,StadiumType,Turf,GameWeather,Temperature,Humidity,WindSpeed,WindDirection
0,2017090700,20170907000118,away,73.91,34.84,1.69,1.13,0.40,81.99,177.18,...,1,Gillette Stadium,"Foxborough, MA",Outdoor,Field Turf,Clear and warm,63.0,77.0,8.0,SW
1,2017090700,20170907000118,away,74.67,32.64,0.42,1.35,0.01,27.61,198.70,...,1,Gillette Stadium,"Foxborough, MA",Outdoor,Field Turf,Clear and warm,63.0,77.0,8.0,SW
2,2017090700,20170907000118,away,74.00,33.20,1.22,0.59,0.31,3.01,202.73,...,1,Gillette Stadium,"Foxborough, MA",Outdoor,Field Turf,Clear and warm,63.0,77.0,8.0,SW
3,2017090700,20170907000118,away,71.46,27.70,0.42,0.54,0.02,359.77,105.64,...,1,Gillette Stadium,"Foxborough, MA",Outdoor,Field Turf,Clear and warm,63.0,77.0,8.0,SW
4,2017090700,20170907000118,away,69.32,35.42,1.82,2.43,0.16,12.63,164.31,...,1,Gillette Stadium,"Foxborough, MA",Outdoor,Field Turf,Clear and warm,63.0,77.0,8.0,SW


In [5]:
df.columns

Index(['GameId', 'PlayId', 'Team', 'X', 'Y', 'S', 'A', 'Dis', 'Orientation',
       'Dir', 'NflId', 'DisplayName', 'JerseyNumber', 'Season', 'YardLine',
       'Quarter', 'GameClock', 'PossessionTeam', 'Down', 'Distance',
       'FieldPosition', 'HomeScoreBeforePlay', 'VisitorScoreBeforePlay',
       'NflIdRusher', 'OffenseFormation', 'OffensePersonnel',
       'DefendersInTheBox', 'DefensePersonnel', 'PlayDirection', 'TimeHandoff',
       'TimeSnap', 'Yards', 'PlayerHeight', 'PlayerWeight', 'PlayerBirthDate',
       'PlayerCollegeName', 'Position', 'HomeTeamAbbr', 'VisitorTeamAbbr',
       'Week', 'Stadium', 'Location', 'StadiumType', 'Turf', 'GameWeather',
       'Temperature', 'Humidity', 'WindSpeed', 'WindDirection'],
      dtype='object')

In [6]:
players_per_play = df.groupby("PlayId").size()
print("Players on field:")
print(players_per_play.value_counts())

Players on field:
22    31007
Name: count, dtype: int64


In [7]:
print("Feature                     Type")
print(df.dtypes)

Feature                     Type
GameId                      int64
PlayId                      int64
Team                       object
X                         float64
Y                         float64
S                         float64
A                         float64
Dis                       float64
Orientation               float64
Dir                       float64
NflId                       int64
DisplayName                object
JerseyNumber                int64
Season                      int64
YardLine                    int64
Quarter                     int64
GameClock                  object
PossessionTeam             object
Down                        int64
Distance                    int64
FieldPosition              object
HomeScoreBeforePlay         int64
VisitorScoreBeforePlay      int64
NflIdRusher                 int64
OffenseFormation           object
OffensePersonnel           object
DefendersInTheBox         float64
DefensePersonnel           object
PlayDirection  

In [8]:
print("\nMissings (% per column)")
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
print(missing_pct[missing_pct > 0].round(2))
#that's pretty convenient since i didn't plan on keeping any of those infos


Missings (% per column)
WindDirection        15.34
WindSpeed            13.47
Temperature           9.33
GameWeather           8.82
StadiumType           6.11
FieldPosition         1.26
Humidity              0.90
OffenseFormation      0.01
Dir                   0.00
Orientation           0.00
DefendersInTheBox     0.00
dtype: float64


In [9]:
print("\nCardinality of Categorical Data")
for col in df.select_dtypes(include="object").columns:
    print(f"  {col}: {df[col].nunique()} unique opts")


Cardinality of Categorical Data
  Team: 2 unique opts
  DisplayName: 2568 unique opts
  GameClock: 901 unique opts
  PossessionTeam: 32 unique opts
  FieldPosition: 32 unique opts
  OffenseFormation: 8 unique opts
  OffensePersonnel: 61 unique opts
  DefensePersonnel: 45 unique opts
  PlayDirection: 2 unique opts
  TimeHandoff: 30709 unique opts
  TimeSnap: 30721 unique opts
  PlayerHeight: 16 unique opts
  PlayerBirthDate: 1897 unique opts
  PlayerCollegeName: 314 unique opts
  Position: 25 unique opts
  HomeTeamAbbr: 32 unique opts
  VisitorTeamAbbr: 32 unique opts
  Stadium: 61 unique opts
  Location: 67 unique opts
  StadiumType: 33 unique opts
  Turf: 23 unique opts
  GameWeather: 73 unique opts
  WindSpeed: 69 unique opts
  WindDirection: 58 unique opts


##Pre-processing

In [10]:
#now I will move the target column to a separate dataframe and remove it from the features one to avoid any leaking
df_target = (
    df.groupby("PlayId")
      .agg(GameId=("GameId", "first"),
           NflIdRusher=("NflIdRusher", "first"),
           Yards=("Yards", "first"))
      .reset_index())
df = df.drop(columns=["Yards"])

print("Target's shape:", df_target.shape)
print(df_target.head())

print("\nYard's Distribution")
print(df_target["Yards"].describe())
print(f"\n% Yards E [-2, 5]: {((df_target['Yards'] >= -2) & (df_target['Yards'] <= 5)).mean()*100:.1f}%")
print(f"% Yards >= 20: {(df_target['Yards'] >= 20).mean()*100:.1f}%")


Target's shape: (31007, 4)
           PlayId      GameId  NflIdRusher  Yards
0  20170907000118  2017090700      2543773      8
1  20170907000139  2017090700      2543773      3
2  20170907000189  2017090700      2543773      5
3  20170907000345  2017090700      2539663      2
4  20170907000395  2017090700      2557917      7

Yard's Distribution
count    31007.000000
mean         4.227626
std          6.449966
min        -15.000000
25%          1.000000
50%          3.000000
75%          6.000000
max         99.000000
Name: Yards, dtype: float64

% Yards E [-2, 5]: 70.8%
% Yards >= 20: 2.5%


In [11]:
COLS_TO_DROP = [
    #as a project's decision, we chose not to explore features related to external conditions
    "GameWeather", "Temperature", "Humidity",
    "WindSpeed", "WindDirection",
    "StadiumType", "Turf", "Location", "Stadium",
    #redundant textual identifiers
    "DisplayName", "JerseyNumber",
    "PlayerCollegeName",
    "HomeTeamAbbr", "VisitorTeamAbbr",
    #not needed, we'll extract it directly
    "TimeSnap",]

df = df.drop(columns=[c for c in COLS_TO_DROP if c in df.columns])
print("New shape:", df.shape)
print("Columns left:", df.columns.tolist())

New shape: (682154, 33)
Columns left: ['GameId', 'PlayId', 'Team', 'X', 'Y', 'S', 'A', 'Dis', 'Orientation', 'Dir', 'NflId', 'Season', 'YardLine', 'Quarter', 'GameClock', 'PossessionTeam', 'Down', 'Distance', 'FieldPosition', 'HomeScoreBeforePlay', 'VisitorScoreBeforePlay', 'NflIdRusher', 'OffenseFormation', 'OffensePersonnel', 'DefendersInTheBox', 'DefensePersonnel', 'PlayDirection', 'TimeHandoff', 'PlayerHeight', 'PlayerWeight', 'PlayerBirthDate', 'Position', 'Week']


In [12]:
#now, starting to work on our actual features, let's do some type conversions
def height_to_inches(h):
    if pd.isna(h):
        return np.nan
    try:
        feet, inches = str(h).split("-")
        return int(feet)*12+int(inches)
    except (ValueError, AttributeError):
        return np.nan

In [13]:
df["PlayerHeight"] = df["PlayerHeight"].apply(height_to_inches)
df["PlayerBirthDate"] = pd.to_datetime(df["PlayerBirthDate"], errors="coerce")
df["TimeHandoff"] = pd.to_datetime(df["TimeHandoff"], errors="coerce", utc=True)
df["PlayerAge"] = ((df["TimeHandoff"].dt.tz_localize(None) - df["PlayerBirthDate"]).dt.days / 365.25)
#checking
print(df[["PlayerHeight", "PlayerWeight", "PlayerAge"]].describe())

        PlayerHeight   PlayerWeight      PlayerAge
count  682154.000000  682154.000000  682154.000000
mean       74.389532     253.414628      26.915541
std         2.595915      48.469869       3.271755
min        66.000000     153.000000      20.355921
25%        72.000000     210.000000      24.424367
50%        75.000000     245.000000      26.447639
75%        76.000000     305.000000      28.856947
max        81.000000     380.000000      42.310746


In [14]:
#all seems ok so:
df = df.drop(columns=["TimeHandoff", "PlayerBirthDate"])

In [15]:
def gameclock_to_pct_elapsed(gc):
#Returns the fraction of the quarter that has already elapsed (0.0 = start, 1.0 = end)
    if pd.isna(gc):
        return np.nan
    parts = str(gc).split(":")
    try:
        minutes = int(parts[0])
        seconds = int(parts[1])
        remaining = minutes * 60 + seconds   #math's  based on seconds to make it easier 
        elapsed = 900 - remaining            
        return elapsed / 900
    except (ValueError, IndexError):
        return np.nan

df["GameClockPct"] = df["GameClock"].apply(gameclock_to_pct_elapsed)
df = df.drop(columns=["GameClock"])
print(df["GameClockPct"].describe())

count    682154.000000
mean          0.484146
std           0.292653
min           0.000000
25%           0.228889
50%           0.486667
75%           0.741111
max           1.000000
Name: GameClockPct, dtype: float64


In [16]:
#i'll apply the same logic to the whole game and then exclude wuarter since it'll become redundant
df["GameElapsedPct"] = ((df["Quarter"] - 1) + df["GameClockPct"]) / 4
df = df.drop(columns=["Quarter"])

In [17]:
#field positions will be converted to x axis (non symmetrical + monotonically growing)
def compute_abs_yardline(row):
#120 yard field (10 end zone + 100 + 10 end zone).
#If PossessionTeam == FieldPosition: ball on own side -> YardLine + 10
#If different: advanced to the opponent's side -> 110 - YardLine
#If on the 50 line: 60 (center of the field)
    if row["YardLine"] == 50:
        return 60.0
    if pd.isna(row["FieldPosition"]):
        return 60.0
    if row["PossessionTeam"] == row["FieldPosition"]:
        return row["YardLine"] + 10
    return 110 - row["YardLine"]

df["AbsYardLine"] = df.apply(compute_abs_yardline, axis=1)
print(df[["YardLine", "FieldPosition", "PossessionTeam", "AbsYardLine"]].tail(10))

        YardLine FieldPosition PossessionTeam  AbsYardLine
682144        38           BLT            BLT         48.0
682145        38           BLT            BLT         48.0
682146        38           BLT            BLT         48.0
682147        38           BLT            BLT         48.0
682148        38           BLT            BLT         48.0
682149        38           BLT            BLT         48.0
682150        38           BLT            BLT         48.0
682151        38           BLT            BLT         48.0
682152        38           BLT            BLT         48.0
682153        38           BLT            BLT         48.0


In [18]:
mask_2017 = df["Season"] == 2017
df.loc[mask_2017, "Orientation"] = (df.loc[mask_2017, "Orientation"] - 90) % 360

In [19]:
is_left = df["PlayDirection"] == "left"
print(f"Rows PlayDirection=='left': {is_left.sum()} ({is_left.mean()*100:.1f}%)")
#flips coordinates and angles to fixed orientation adopted
df.loc[is_left, "X"] = 120 - df.loc[is_left, "X"]
df.loc[is_left, "Y"] = 53.3 - df.loc[is_left, "Y"]
df.loc[is_left, "Dir"] = (df.loc[is_left, "Dir"] + 180) % 360
df.loc[is_left, "Orientation"] = (df.loc[is_left, "Orientation"] + 180) % 360
df.loc[is_left, "AbsYardLine"] = 120 - df.loc[is_left, "AbsYardLine"]

df["PlayDirection"] = "right"   #any game: left->right
df["X_rel"] = df["X"] - df["AbsYardLine"] # X=0 at LOS, pos = attack dir
df["Y_rel"] = df["Y"] - (53.3 / 2) # Y=0 at field center
print(df[["X", "Y", "AbsYardLine", "X_rel", "Y_rel"]].describe())

Rows PlayDirection=='left': 344410 (50.5%)
                   X              Y    AbsYardLine          X_rel  \
count  682154.000000  682154.000000  682154.000000  682154.000000   
mean       59.261724      26.789567      60.082175      -0.820451   
std        25.488647       7.192956      25.297636      36.039670   
min         2.350000       2.690000      11.000000    -105.840000   
25%        38.070000      22.690000      39.000000     -11.210000   
50%        55.510000      26.720000      60.000000      -0.240000   
75%        79.270000      30.790000      81.000000       5.930000   
max       119.340000      56.450000     109.000000     108.340000   

               Y_rel  
count  682154.000000  
mean        0.139567  
std         7.192956  
min       -23.960000  
25%        -3.960000  
50%         0.070000  
75%         4.140000  
max        29.800000  


In [20]:
#all seems ok so:
df = df.drop(columns=["PlayDirection"])

In [21]:
#speed
df["S"] = df["Dis"] * 10 #freq os frame collection was 10hz
print(df[["S", "Dis"]].describe())
#there's a point: since S is noisy, i'd expect A to also be. thus, i would need to correct it as well. however, this dataset does not provide access to consectuive frames such that we could evaluate acceleration numerically (A = (S[t] - S[t-1]) / 0.1). so, i had to download the whole NGS dataset (which does provide all frames in every game), retrieve the frame immediatly before it, calculate the gap distance and then correct A. Deploy-wise, it's important to notice that the model is nt relying 100% on *ONLY* the handoff moment (N-th frame), but on the (N-1, N) frame pair. HOWEVER, again, those complete data were not set public. so i couldn't do that and had to keep A as it is and hope the noise level wouldn't hurt the performance that much lol

                   S            Dis
count  682154.000000  682154.000000
mean        2.781932       0.278193
std         1.452182       0.145218
min         0.000000       0.000000
25%         1.600000       0.160000
50%         2.700000       0.270000
75%         3.900000       0.390000
max        13.900000       1.390000


In [22]:
#decomposition of angles
#Dir and Orientation: angles in degrees, 0 = north, clockwise direction.
#converted to unit vector (sin, cos) for the model
dir_rad = np.deg2rad(90 - df["Dir"])
ori_rad = np.deg2rad(90 - df["Orientation"])
df["Dir_sin"] = np.sin(dir_rad)
df["Dir_cos"] = np.cos(dir_rad)
df["Ori_sin"] = np.sin(ori_rad)
df["Ori_cos"] = np.cos(ori_rad)
#vector componentes for velocity of movement
df["Sx"] = df["S"] * df["Dir_cos"]
df["Sy"] = df["S"] * df["Dir_sin"]

In [23]:
df["IsRusher"] = (df["NflId"] == df["NflIdRusher"]).astype(int)
#Player side: Offensive if Team matches PossessionTeam
#(Team is 'home'/'away'; find out which team is in possession)
#Reconstructed via NflIdRusher: the rusher's team is the offense.
rusher_team = (df[df["IsRusher"] == 1][["PlayId", "Team"]].rename(columns={"Team": "OffenseTeamSide"}))
df = df.merge(rusher_team, on="PlayId", how="left")
df["IsOffense"] = (df["Team"] == df["OffenseTeamSide"]).astype(int)
df = df.drop(columns=["OffenseTeamSide"])
#expected 11 offense and 11 defense
counts = df.groupby("PlayId")["IsOffense"].sum()
print("Players Distribution")
print(counts.value_counts())

Players Distribution
IsOffense
11    31007
Name: count, dtype: int64


In [24]:
# Cats: OffenseFormation, OffensePersonnel, DefensePersonnel, Position, PossessionTeam, FieldPosition, Team
cat_cols = ["OffenseFormation", "OffensePersonnel", "DefensePersonnel",
            "Position", "PossessionTeam", "FieldPosition", "Team"]
cat_cols = [c for c in cat_cols if c in df.columns]
enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
df[cat_cols] = enc.fit_transform(df[cat_cols].astype(str))

#saving parameters for testing later
encoders = {"ordinal": enc, "cat_cols": cat_cols}

In [25]:
df.columns

Index(['GameId', 'PlayId', 'Team', 'X', 'Y', 'S', 'A', 'Dis', 'Orientation',
       'Dir', 'NflId', 'Season', 'YardLine', 'PossessionTeam', 'Down',
       'Distance', 'FieldPosition', 'HomeScoreBeforePlay',
       'VisitorScoreBeforePlay', 'NflIdRusher', 'OffenseFormation',
       'OffensePersonnel', 'DefendersInTheBox', 'DefensePersonnel',
       'PlayerHeight', 'PlayerWeight', 'Position', 'Week', 'PlayerAge',
       'GameClockPct', 'GameElapsedPct', 'AbsYardLine', 'X_rel', 'Y_rel',
       'Dir_sin', 'Dir_cos', 'Ori_sin', 'Ori_cos', 'Sx', 'Sy', 'IsRusher',
       'IsOffense'],
      dtype='object')

In [26]:
#player features 
PLAYER_FEATURES = [
    "X_rel", "Y_rel", "S", "A", "Dis",
    "Dir_sin", "Dir_cos", "Ori_sin", "Ori_cos",
    "Sx", "Sy",
    "PlayerHeight", "PlayerWeight", "PlayerAge",
    "Position",]
#game features 
PLAY_FEATURES = [
    "GameId",
    "Season", "Week", "Quarter", "Down", "Distance",
    "HomeScoreBeforePlay", "VisitorScoreBeforePlay",
    "OffenseFormation", "OffensePersonnel", "DefensePersonnel",
    "DefendersInTheBox", "PossessionTeam", "FieldPosition",
    "AbsYardLine",
    "GameClockPct",]


def flatten_play(group):
    rusher  = group[group["IsRusher"] == 1]
    offense = group[(group["IsRusher"] == 0) & (group["IsOffense"] == 1)]
    defense = group[group["IsOffense"] == 0]
    ordered = pd.concat([rusher, offense, defense]).reset_index(drop=True)

    row = {}
    for f in PLAY_FEATURES:
        if f in group.columns:
            row[f] = group.iloc[0][f]
    for i, (_, player) in enumerate(ordered.iterrows()):
        if i == 0:
            prefix = "rusher"
        elif i <= 10:
            prefix = f"off{i}"
        else:
            prefix = f"def{i - 10}"
        for f in PLAYER_FEATURES:
            row[f"{prefix}_{f}"] = player[f]
    return pd.Series(row)


df_training = df.groupby("PlayId").apply(flatten_play).reset_index()
print(df_training.shape)
df_training.head()

(31007, 346)


/var/folders/qb/ldt_bc3j77969pkfrmpl2bbc0000gn/T/ipykernel_22402/3479914705.py:42: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_training = df.groupby("PlayId").apply(flatten_play).reset_index()


,PlayId,GameId,Season,Week,Down,Distance,HomeScoreBeforePlay,VisitorScoreBeforePlay,OffenseFormation,OffensePersonnel,...,def11_Dir_sin,def11_Dir_cos,def11_Ori_sin,def11_Ori_cos,def11_Sx,def11_Sy,def11_PlayerHeight,def11_PlayerWeight,def11_PlayerAge,def11_Position
0,20170907000118,2.017091e+09,2017.0,1.0,3.0,2.0,0.0,0.0,5.0,9.0,...,-0.072194,0.997391,0.299374,0.954136,0.099739,-0.007219,78.0,308.0,23.184120,5.0
1,20170907000139,2.017091e+09,2017.0,1.0,1.0,10.0,0.0,0.0,5.0,9.0,...,-0.904381,-0.426727,-0.119790,0.992799,-0.554745,-1.175695,78.0,308.0,23.184120,5.0
2,20170907000189,2.017091e+09,2017.0,1.0,1.0,10.0,0.0,0.0,6.0,9.0,...,0.990122,-0.140210,0.001222,0.999999,-0.420630,2.970365,78.0,308.0,23.184120,5.0
3,20170907000345,2.017091e+09,2017.0,1.0,2.0,2.0,0.0,0.0,3.0,53.0,...,-0.843391,-0.537300,0.365852,0.930673,-1.343249,-2.108479,74.0,307.0,24.183436,5.0
4,20170907000395,2.017091e+09,2017.0,1.0,1.0,10.0,7.0,0.0,5.0,18.0,...,-0.972897,-0.231239,-0.028794,0.999585,-0.763087,-3.210560,74.0,320.0,23.597536,5.0


In [28]:
df_training_unflattened = df.merge(
    df_target[["PlayId", "Yards"]],
    on="PlayId",
    how="inner")
df_training_unflattened.to_parquet("df_training_unflattened.parquet", index=False)